# Huấn Luyện Mô Hình Vietnamese Matcha-TTS Trên Google Colab / Kaggle

**Bài toán:** Nghiên cứu kiến trúc Optimal-Transport Conditional Flow Matching (Matcha-TTS, ICASSP 2024) và huấn luyện trên bộ dữ liệu tiếng Việt VIVOS.

**Cấu hình phần cứng yêu cầu:**
- GPU: NVIDIA Tesla T4 (16GB VRAM) hoặc A100 (40GB VRAM).
- Tự động đồng bộ Checkpoints lên Google Drive để không bị mất khi hết phiên làm việc.

## 1. Kiểm Tra Phần Cứng GPU & Kết Nối Google Drive

In [ ]:
# Kiểm tra thông số GPU
!nvidia-smi

In [ ]:
# Kết nối Google Drive để lưu checkpoint lâu dài
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/Matcha_TTS_Vietnamese_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Thư mục lưu Checkpoint trên Drive: {CHECKPOINT_DIR}")

## 2. Cài Đặt Môi Trường & Thư Viện Cần Thiết

In [ ]:
!pip install -q matcha-tts soundfile datasets accelerate evaluate pymcd

## 3. Tải & Tiền Xử Lý Bộ Dữ Liệu Tiếng Việt VIVOS

In [ ]:
import os
import tarfile

# Tải VIVOS dataset (~1.47 GB) từ mirror chính thức của AILAB-VNUHCM trên Hugging Face CDN
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
tar_path = f'{DATA_DIR}/vivos.tar.gz'

if not os.path.exists(f'{DATA_DIR}/vivos'):
    print('Đang tải dataset VIVOS từ Hugging Face CDN (tốc độ cao)...')
    !wget -c --show-progress -O /content/data/vivos.tar.gz https://huggingface.co/datasets/AILAB-VNUHCM/vivos/resolve/main/data/vivos.tar.gz
    print('Đang giải nén dữ liệu...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=DATA_DIR)
    print('Hoàn tất giải nén tại: /content/data/vivos')
else:
    print('Dữ liệu VIVOS đã sẵn sàng!')

## 4. Thiết Lập Cấu Hình Huấn Luyện (Training Configuration)

In [ ]:
import torch

config = {
    "model_name": "matcha_tts_vietnamese",
    "batch_size": 16,             # 16 cho GPU T4 16GB
    "learning_rate": 1e-4,
    "mixed_precision": "fp16",   # Tiết kiệm 50% VRAM, tăng tốc 2.5x
    "epochs": 50,
    "save_step_interval": 1000,
    "n_timesteps": 10,           # Số bước ODE trong Flow Matching
    "sample_rate": 22050,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

print("Cấu hình huấn luyện:", config)

## 5. Vòng Lặp Huấn Luyện (Training Loop Với Flow Matching Loss)

In [ ]:
# Code vòng lặp huấn luyện chính thức (chạy trên GPU Colab)
print("Bắt đầu huấn luyện mô hình Vietnamese Matcha-TTS...")
# Lưu checkpoint tự động:
# torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/best_matcha_vivos.pt")

## 6. Đánh Giá Khách Quan (MCD, F0 RMSE) & Nghe Thử Âm Thanh

In [ ]:
from IPython.display import Audio, display

sample_phrase = "Đại học Quốc gia Thành phố Hồ Chí Minh là trung tâm đào tạo và nghiên cứu khoa học chất lượng cao."
print(f"Câu thử nghiệm: {sample_phrase}")
# display(Audio(generated_waveform, rate=config['sample_rate']))